<a href="https://colab.research.google.com/github/hassanmufti2002/mini-project-data-cleaning/blob/main/Data_Cleaning_Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re

# Load - this is your file name from screenshot
df = pd.read_csv('Messy_data.csv')

print("Original:", df.shape)
print(df.head())
print(df.columns.tolist())
print(df.isnull().sum())

# Fix spaces + text - delhi, DELHI -> Delhi
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

if 'City' in df.columns:
    df['City'] = df['City'].str.lower().str.title().replace('Nan','Unknown')
if 'OrderStatus' in df.columns:
    df['OrderStatus'] = df['OrderStatus'].str.lower().str.title().replace('Nan','Pending')
if 'CustomerName' in df.columns:
    df['CustomerName'] = df['CustomerName'].replace('nan','').replace('','Unknown Customer')

# Fix data errors - three, 2 pcs, 18,500, unknown, forty
word_map = {'one':1,'two':2,'three':3,'four':4,'five':5,'forty':40,'thirty':30}
def clean_num(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().replace(',', '').strip()
    if s in ['unknown','nan','']: return np.nan
    if s in word_map: return word_map[s]
    m = re.findall(r'\d+\.?\d*', s)
    return float(m[0]) if m else np.nan

for c in ['Quantity','UnitPrice','CustomerAge']:
    if c in df.columns:
        df[c] = df[c].apply(clean_num)

if 'OrderDate' in df.columns:
    df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce', dayfirst=True)

# Missing + Outliers - Day 4 capping method
for c in ['Quantity','UnitPrice','CustomerAge']:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].median())

for c in ['Quantity','UnitPrice']:
    if c in df.columns:
        Q1,Q3 = df[c].quantile(0.25), df[c].quantile(0.75)
        upper = Q3 + 1.5*(Q3-Q1)
        df[c] = df[c].clip(upper=upper)

if 'CustomerAge' in df.columns:
    df['CustomerAge'] = df['CustomerAge'].clip(5,90)

# Duplicates - 5 duplicates
print(f"Before dedup: {df.shape}, dups: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"After dedup: {df.shape}")

# Convert types
if 'Quantity' in df.columns: df['Quantity'] = df['Quantity'].astype(int)
if 'CustomerAge' in df.columns: df['CustomerAge'] = df['CustomerAge'].astype(int)

# TotalAmount = quantity x unit price
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

# CREATE FINAL FILE - Ma'am wants this name
df.to_csv('cleaned_sales_dataset.csv', index=False)
print("\n✓ FILE CREATED: cleaned_sales_dataset.csv")
print(df.head())
print(df.shape)

Original: (185, 9)
   OrderID  CustomerName     City     Product Quantity UnitPrice   OrderDate  \
0    10001   Nikhil Jain   Mumbai      Laptop      3.0  50790.72  2026-01-18   
1    10002   Simran Kaur   delhi      Monitor      1.0  10307.27  2026-01-28   
2    10003      Zoya Ali    Delhi  Headphones      5.0   2439.64  2026-02-27   
3    10004  Priya Bansal    Delhi      Tablet      4.0  20945.65  02/03/2026   
4    10005           NaN   Mumbai       Phone      4.0  15822.27  2026-02-14   

  OrderStatus CustomerAge  
0   Completed        61.0  
1     Pending        50.0  
2   Delivered        35.0  
3     Pending        66.0  
4   Delivered        34.0  
['OrderID', 'CustomerName', 'City', 'Product', 'Quantity', 'UnitPrice', 'OrderDate', 'OrderStatus', 'CustomerAge']
OrderID         0
CustomerName    2
City            2
Product         0
Quantity        2
UnitPrice       2
OrderDate       2
OrderStatus     2
CustomerAge     2
dtype: int64
Before dedup: (185, 9), dups: 5
After dedu

/tmp/ipykernel_1341/1511777141.py:39: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce', dayfirst=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import files
files.download('cleaned_sales_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('cleaned_sales_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>